# Gazette Iteration Pipeline

This notebook builds a repeatable pipeline:
1. Load Mistral JSON
2. Join pages by index -> markdown
3. Write combined markdown
4. Parse markdown into final JSON envelope


In [1]:
from __future__ import annotations
import json
import re
from datetime import datetime, timezone
from pathlib import Path

INPUT_JSON = Path(r"C:\Users\Ronald Wahome\Documents\gazette_iteration_pipeline\gazette_1999-12-03_80.raw.json")


def _output_base_name(path: Path) -> str:
    name = path.name
    if name.endswith(".raw.json"):
        return name[: -len(".raw.json")]
    if name.lower().endswith(".json"):
        return path.stem
    return path.stem


_base = _output_base_name(INPUT_JSON)
_out_dir = INPUT_JSON.parent
JOINED_MD_OUT = _out_dir / f"{_base}_joined_from_json.md"
FINAL_JSON_OUT = _out_dir / f"{_base}_final_envelope.json"
INPUT_JSON, JOINED_MD_OUT, FINAL_JSON_OUT


(WindowsPath('C:/Users/Ronald Wahome/Documents/gazette_iteration_pipeline/gazette_1999-12-03_80.raw.json'),
 WindowsPath('C:/Users/Ronald Wahome/Documents/gazette_iteration_pipeline/gazette_1999-12-03_80_joined_from_json.md'),
 WindowsPath('C:/Users/Ronald Wahome/Documents/gazette_iteration_pipeline/gazette_1999-12-03_80_final_envelope.json'))

In [2]:
def load_pages(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f"No such file: {path}")
    raw = path.read_text(encoding="utf-8")
    if not raw.strip():
        raise ValueError(
            f"Input file is empty ({path.stat().st_size} bytes): {path}\n"
            "Re-export the Mistral OCR JSON or set INPUT_JSON to a non-empty .raw.json."
        )
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path}: {e}") from e

    # Mistral batch export: [{ "pdf_url", "id", "pages": [...] }, ...] — merge every block.
    # Single envelope: {"pages": [...]} or legacy [{ "pages": [...] }] with one item.
    # Legacy page list: [{"index": 0, "markdown": "..."}, ...]
    blocks: list[dict] = []
    if isinstance(payload, dict) and "pages" in payload:
        blocks = [payload]
    elif isinstance(payload, list):
        if payload and all(isinstance(x, dict) for x in payload) and any("pages" in x for x in payload):
            blocks = [x for x in payload if isinstance(x, dict) and "pages" in x]
        else:
            blocks = [{"pages": payload, "pdf_url": None, "id": None}]
    else:
        raise ValueError(f"Unsupported JSON root type: {type(payload).__name__}")

    def _page_sort_key(p: dict):
        i = p.get("index")
        try:
            return int(i)
        except (TypeError, ValueError):
            return 0

    rows: list[dict] = []
    global_index = 0
    for block in blocks:
        pages = [p for p in (block.get("pages") or []) if isinstance(p, dict)]
        pages.sort(key=_page_sort_key)
        for p in pages:
            if not isinstance(p, dict):
                continue
            orig = p.get("index")
            md = p.get("markdown", "") or ""
            if orig is None:
                continue
            try:
                int(orig)
            except (TypeError, ValueError):
                continue
            rows.append({
                "index": global_index,
                "markdown": md,
                "original_page_index": int(orig),
                "pdf_url": block.get("pdf_url"),
                "mistral_doc_id": block.get("id"),
            })
            global_index += 1

    return rows

pages = load_pages(INPUT_JSON)
if not pages:
    print("pages: 0")
else:
    n_docs = len({p.get("mistral_doc_id") for p in pages if p.get("mistral_doc_id") is not None})
    print("pages:", len(pages), "range:", pages[0]["index"], "->", pages[-1]["index"], end="")
    if n_docs:
        print("  mistral_doc_ids:", n_docs)
    else:
        print()


pages: 60 range: 0 -> 59


In [3]:
def join_pages_to_markdown(pages, add_index_headers=True):
    chunks = []
    for p in pages:
        idx = p["index"]
        md = p["markdown"].strip()
        if add_index_headers:
            chunks.append(f"\n\n---\n\n## Index {idx}\n\n{md}\n")
        else:
            chunks.append(md)
    return "\n".join(chunks).strip() + "\n"

joined_markdown = join_pages_to_markdown(pages, add_index_headers=True)
JOINED_MD_OUT.write_text(joined_markdown, encoding="utf-8")
print("wrote:", JOINED_MD_OUT)


wrote: C:\Users\Ronald Wahome\Documents\gazette_iteration_pipeline\gazette_1999-12-03_80_joined_from_json.md


In [5]:
NOTICE_SPLIT_RE = re.compile(r"(?=(?:GAZETTE|GRZETTE)\s+NOTICE\s+NO\.?\s*\d+)", re.IGNORECASE)
NOTICE_NO_RE = re.compile(r"(?:GAZETTE|GRZETTE)\s+NOTICE\s+NO\.?\s*(\d+)", re.IGNORECASE)
DATE_RE = re.compile(r"\b\d{1,2}(?:st|nd|rd|th)\s+[A-Za-z]+,\s+\d{4}\b")


def _split_table_row(line: str):
    row = line.strip()
    if not row.startswith("|"):
        return []
    row = row[1:-1] if row.endswith("|") else row[1:]
    return [cell.strip() for cell in row.split("|")]


def _is_separator_row(line: str):
    cells = _split_table_row(line)
    if not cells:
        return False
    return all(
        not cell.strip() or re.fullmatch(r":?-{3,}:?", cell.strip().replace(" ", ""))
        for cell in cells
    )


def _normalize_key(value: str):
    return re.sub(r"[^a-z0-9]+", "", value.lower())


def _normalize_row(cells, width: int):
    cells = list(cells)
    if len(cells) < width:
        cells.extend([""] * (width - len(cells)))
    elif len(cells) > width:
        # OCR noise sometimes creates extra pipes; keep width stable.
        cells = cells[: width - 1] + [" | ".join(cells[width - 1:])]
    return cells


def _normalize_headers(headers, width: int):
    headers = _normalize_row(headers, width)
    normalized = []
    for i, h in enumerate(headers, start=1):
        label = h.strip()
        normalized.append(label if label else f"column_{i}")
    return normalized


def _is_duplicate_header(row, headers):
    return [_normalize_key(cell) for cell in row] == [_normalize_key(cell) for cell in headers]


def _merge_sparse_row(rows, row):
    non_empty_indexes = [i for i, cell in enumerate(row) if cell.strip()]
    if not rows or len(non_empty_indexes) != 1:
        return False

    idx = non_empty_indexes[0]
    if idx == 0:
        return False

    addition = row[idx].strip()
    rows[-1][idx] = f"{rows[-1][idx].rstrip()} {addition}".strip()
    return True


def _append_continuation(rows, line: str):
    if not rows:
        return False

    text = line.strip()
    if not text:
        return False
    if text.startswith(("#", "---", "## Index")) or NOTICE_NO_RE.search(text):
        return False

    # Mistral often wraps a table cell onto a plain text line, especially bullet cells.
    if not (text.startswith(("•", "- ", "* ", "(", "and ", "or ")) or line[:1].isspace()):
        return False

    target_idx = next((i for i in range(len(rows[-1]) - 1, -1, -1) if rows[-1][i].strip()), len(rows[-1]) - 1)
    separator = "\n" if text.startswith(("•", "- ", "* ", "(")) else " "
    rows[-1][target_idx] = f"{rows[-1][target_idx].rstrip()}{separator}{text}".strip()
    return True


def _records_from_rows(headers, rows):
    return [dict(zip(headers, row)) for row in rows]


def extract_tables_from_markdown(text: str):
    lines = text.splitlines()
    tables = []
    i = 0

    while i < len(lines) - 1:
        header_line = lines[i].rstrip()
        sep_line = lines[i + 1].rstrip()

        if not (
            header_line.lstrip().startswith("|")
            and sep_line.lstrip().startswith("|")
            and _is_separator_row(sep_line)
        ):
            i += 1
            continue

        raw_headers = _split_table_row(header_line)
        sep_cells = _split_table_row(sep_line)
        width = max(len(raw_headers), len(sep_cells), 1)
        headers = _normalize_headers(raw_headers, width)

        raw_table_lines = [header_line, sep_line]
        rows = []
        i += 2

        while i < len(lines):
            row_line = lines[i].rstrip()
            stripped = row_line.strip()

            if row_line.lstrip().startswith("|"):
                raw_table_lines.append(row_line)
                if _is_separator_row(row_line):
                    i += 1
                    continue

                row = _normalize_row(_split_table_row(row_line), width)
                if _is_duplicate_header(row, headers):
                    i += 1
                    continue
                if any(cell.strip() for cell in row) and not _merge_sparse_row(rows, row):
                    rows.append(row)
                i += 1
                continue

            if _append_continuation(rows, row_line):
                raw_table_lines.append(row_line)
                i += 1
                continue

            if not stripped:
                i += 1
            break

        tables.append({
            "source": "markdown_table_heuristic",
            "column_count": width,
            "headers": headers,
            "rows": rows,
            "records": _records_from_rows(headers, rows),
            "raw_table_markdown": "\n".join(raw_table_lines),
        })

    return tables


def markdown_to_envelope(markdown_text: str):
    parts = [p.strip() for p in NOTICE_SPLIT_RE.split(markdown_text) if p.strip()]
    notices = []
    for part in parts:
        m = NOTICE_NO_RE.search(part)
        if not m:
            continue

        tables = extract_tables_from_markdown(part)
        notices.append({
            "notice_no": m.group(1),
            "dates_found": DATE_RE.findall(part),
            "table_count": len(tables),
            "tables": tables,
            "raw_markdown": part,
        })

    return {
        "pipeline": "raw-mistral-json -> joined-markdown -> parsed-envelope",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": {
            "mistral_json": str(INPUT_JSON),
            "joined_markdown": str(JOINED_MD_OUT),
        },
        "stats": {
            "notice_count": len(notices),
            "char_count_markdown": len(markdown_text),
            "notices_with_tables": sum(1 for n in notices if n["table_count"] > 0),
            "total_tables": sum(n["table_count"] for n in notices),
        },
        "notices": notices,
    }

final_envelope = markdown_to_envelope(joined_markdown)
FINAL_JSON_OUT.write_text(json.dumps(final_envelope, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote:", FINAL_JSON_OUT)
print("notices:", final_envelope["stats"]["notice_count"])
print("tables:", final_envelope["stats"]["total_tables"])


wrote: C:\Users\Ronald Wahome\Documents\gazette_iteration_pipeline\gazette_1999-12-03_80_final_envelope.json
notices: 190
tables: 46
